In [0]:
%pip install fitparse
dbutils.library.restartPython()

In [0]:
import zipfile
import os
import io
import json
import urllib.request
from urllib.error import HTTPError
from pathlib import Path
import pandas as pd
from fitparse import FitFile
from utils.config import ATLEET

In [0]:
# 1. Haal je API-sleutel veilig op uit de Databricks kluis
api_key = dbutils.secrets.get(scope="garmin-scope", key="garmin-api-key")

print("Sleutel succesvol geladen. Tijdelijk access token aanvragen...")

# 2. Vraag een token aan bij Garmin
token_url = "https://personal.connectapi.garmin.com/personalapitoken-service/personal-api/personal-api-key/token"
req_token = urllib.request.Request(token_url, method="POST")
req_token.add_header("Authorization", f"ApiKey {api_key}")

try:
    with urllib.request.urlopen(req_token) as response:
        access_token = response.read().decode('utf-8').strip()
        access_token = access_token.strip('"')
        print("Token succesvol ontvangen! Laatste activiteiten ophalen...\n")
        
except HTTPError as e:
    error_details = e.read().decode('utf-8') if e.fp else str(e)
    print(f"Fout bij ophalen token ({e.code}): {error_details}")
    raise e

# 3. Vraag je laatste 7 activiteiten op met het verkregen access token
data_url = "https://personal.connectapi.garmin.com/activitylist-service/activities/search/activities?start=0&limit=7"
req_data = urllib.request.Request(data_url, method="GET")
req_data.add_header("Authorization", f"Bearer {access_token}")
req_data.add_header("Accept", "application/json")

try:
    with urllib.request.urlopen(req_data) as response:
        data = json.loads(response.read().decode('utf-8'))
        print("Jouw 5 laatste trainingen zijn opgehaald")
        
except HTTPError as e:
    error_details = e.read().decode('utf-8') if e.fp else str(e)
    print(f"Fout bij ophalen data ({e.code}): {error_details}")

In [0]:
# Hoofdmap voor de bestanden
base_dir = Path("fit_bestanden_Garmin")

if data and len(data) > 0:
    for activiteit in data:
        activity_id = activiteit.get('activityId')
        
        # Haal de starttijd op
        start_time = activiteit.get('startTimeLocal', '')
        if not start_time:
            print(f"Kan geen starttijd vinden voor {activity_id}, activiteit wordt overgeslagen.")
            continue
            
        # Parse het jaartal, de datum en de tijd
        date_part = start_time[:10]  # yyyy-mm-dd
        year = date_part[:4]         # yyyy
        time_part = start_time[11:19].replace(':', '-') # hh-mm-ss
        
        # Maak een mapstructuur aan per jaar
        year_dir = base_dir / year
        year_dir.mkdir(parents=True, exist_ok=True)
        
        # Definieer de nieuwe bestandsnaam
        file_name = f"{date_part}_{time_part}.fit"
        dbfs_path = year_dir / file_name
        
        # CONTROLE: Bestaat het bestand al? Sla het dan over.
        if dbfs_path.exists():
            print(f"[{date_part}] Bestand '{file_name}' bestaat al. Overslaan.")
            continue
            
        print(f"\nNieuw bestand ophalen: {file_name}...")
        
        # Roep het download-service endpoint aan met je access token
        download_url = f"https://personal.connectapi.garmin.com/download-service/files/activity/{activity_id}"
        req_download = urllib.request.Request(download_url, method="GET")
        req_download.add_header("Authorization", f"Bearer {access_token}")
        
        try:
            with urllib.request.urlopen(req_download) as response:
                file_data = response.read()
                
                # Probeer het FIT-bestand uit de ZIP te halen
                try:
                    with zipfile.ZipFile(io.BytesIO(file_data)) as z:
                        fit_filename = next((name for name in z.namelist() if name.endswith('.fit')), None)
                        
                        if fit_filename:
                            fit_data = z.read(fit_filename)
                        else:
                            print("  -> Waarschuwing: Geen .fit bestand gevonden in de zip. Ruwe bytes opgeslagen.")
                            fit_data = file_data
                except zipfile.BadZipFile:
                    # Als het geen zip is, slaan we direct de ruwe bytes op
                    fit_data = file_data
                    
                # Sla het bestand op in de juiste map
                with open(dbfs_path, "wb") as f:
                    f.write(fit_data)
                    
                print(f"  -> Succesvol opgeslagen in: {dbfs_path}")
                
        except HTTPError as e:
            error_details = e.read().decode('utf-8') if e.fp else str(e)
            print(f"  -> Fout bij downloaden ({e.code}): {error_details}")
else:
    print("Geen activiteiten gevonden in de data lijst.")

In [0]:
base_dir = "fit_bestanden_Garmin/2026"
data = []

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith(".fit"):
            file_path = os.path.join(root, file)
            
            try:
                fitfile = FitFile(file_path)
                
                # 1. Haal de algemene sessie-samenvatting op
                session_data = {}
                for record in fitfile.get_messages('session'):
                    for data_field in record:
                        session_data[data_field.name] = data_field.value
                        
                if session_data:
                    sport = session_data.get("sport")
                    timer_time = session_data.get("total_timer_time", 0)
                    moving_time = session_data.get("total_moving_time")
                    
                    # --- DE DEFINITIEVE ZWEM-FIX: TEL ACTIEVE BAANTJES OP ---
                    if sport == "swimming":
                        actieve_zwemtijd = 0
                        has_lengths = False
                        
                        # Loop door alle losse baantjes (lengths) in het FIT-bestand
                        for length in fitfile.get_messages('length'):
                            has_lengths = True
                            l_type = length.get_value('length_type')
                            
                            # 'active' (of 0) = zwemmen. 'idle' (of 1) = rusten aan de kant.
                            if l_type in ['active', 0]:
                                baantje_tijd = length.get_value('total_timer_time')
                                if baantje_tijd:
                                    actieve_zwemtijd += baantje_tijd
                                    
                        # Als we succesvol de actieve baantjes hebben opgeteld, overschrijven we de moving_time
                        if has_lengths and actieve_zwemtijd > 0:
                            moving_time = actieve_zwemtijd
                    
                    # 2. Bepaal de uiteindelijke actieve tijd voor het dataframe
                    if moving_time and moving_time > 0:
                        actieve_tijd = moving_time
                    else:
                        actieve_tijd = timer_time
                    
                    data.append({
                        "bestandsnaam": file,
                        "datum": session_data.get("start_time"),
                        "sport": sport,
                        "sub_sport": session_data.get("sub_sport"),
                        "tijd_seconden": actieve_tijd, 
                        "tijd_verstreken": session_data.get("total_elapsed_time"),
                        "afstand_meters": session_data.get("total_distance"),
                        "calorieen": session_data.get("total_calories"),
                        "hartslag_gem": session_data.get("avg_heart_rate"),
                        "hartslag_max": session_data.get("max_heart_rate"),
                        "vermogen_gem": session_data.get("avg_power"),
                        "vermogen_np": session_data.get("normalized_power"),
                        "cadans_gem": session_data.get("avg_cadence"),
                        "hoogtemeters": session_data.get("total_ascent"),
                        "tss": session_data.get("training_stress_score"),
                        "intensity_factor": session_data.get("intensity_factor")
                    })
            except Exception as e:
                print(f"Fout bij parsen van {file}: {e}")

df_pandas = pd.DataFrame(data)

In [0]:
def bereken_fallback_tss(row):
    # 1. Bestaande powermeter TSS behouden
    if pd.notnull(row['tss']) and row['tss'] > 0:
        return row['tss']
    
    tijd_uren = row.get('tijd_seconden', 0) / 3600
    if tijd_uren == 0:
        return 0
        
    # 2. hrTSS voor Hardlopen (en wandelen)
    if row['sport'] in ['running', 'walking', 'hiking', 'alpine_skiing', 'backcountry_skiing', 'cross_country_skiing'] and pd.notnull(row['hartslag_gem']):
        tss_factor = row['hartslag_gem'] / ATLEET.LTHR_HARDLOPEN
        return tijd_uren * (tss_factor ** 2) * 100
        
    # 3. hrTSS voor Fietsen (zonder powermeter)
    elif row['sport'] == 'cycling' and pd.notnull(row['hartslag_gem']):
        tss_factor = row['hartslag_gem'] / ATLEET.LTHR_FIETSEN
        return tijd_uren * (tss_factor ** 2) * 100
        
    # 4. sTSS voor Zwemmen (gebaseerd op pace/snelheid)
    elif row['sport'] == 'swimming' and pd.notnull(row['afstand_meters']) and row['afstand_meters'] > 0:
        snelheid_ms = row['afstand_meters'] / row['tijd_seconden']
        tss_factor = snelheid_ms / ATLEET.CSS_ZWEMMEN
        # Waterweerstand neemt kubiek toe, vandaar tot de macht 3 in de zwem-formule
        return tijd_uren * (tss_factor ** 3) * 100  
        
    return 0

# Pas de berekening toe om de lege gaten in de 'tss' kolom te vullen
df_pandas['tss'] = df_pandas.apply(bereken_fallback_tss, axis=1)

# Rond het af op 1 decimalen, wat de standaard is in de sportwetenschap (bijv. 0.8)
df_pandas['tss'] = df_pandas['tss'].round(1)

In [0]:
def bereken_fallback_if(row):
    # 1. Bestaande Garmin IF (uit powermeter) behouden
    if pd.notnull(row['intensity_factor']) and row['intensity_factor'] > 0:
        return row['intensity_factor']
        
    # 2. IF voor Hardlopen, Wandelen en Hiken (op basis van hartslag)
    if row['sport'] in ['running', 'walking', 'hiking'] and pd.notnull(row['hartslag_gem']):
        return row['hartslag_gem'] / ATLEET.LTHR_HARDLOPEN
        
    # 3. IF voor Fietsen (op basis van hartslag, zonder powermeter)
    elif row['sport'] == 'cycling' and pd.notnull(row['hartslag_gem']):
        return row['hartslag_gem'] / ATLEET.LTHR_FIETSEN
        
    # 4. IF voor Zwemmen (op basis van tempo/snelheid)
    elif row['sport'] == 'swimming' and pd.notnull(row['afstand_meters']) and row['tijd_seconden'] > 0:
        snelheid_ms = row['afstand_meters'] / row['tijd_seconden']
        return snelheid_ms / ATLEET.CSS_ZWEMMEN
        
    return None

df_pandas['intensity_factor'] = df_pandas.apply(bereken_fallback_if, axis=1)

# Rond het af op 3 decimalen, wat de standaard is in de sportwetenschap (bijv. 0.852)
df_pandas['intensity_factor'] = df_pandas['intensity_factor'].round(3)


In [0]:
# Sla direct op als parquet bestand in de app-map
output_pad = "triathlon_app/garmin_data.parquet"
df_pandas.to_parquet(output_pad, index=False)
print(f"Data succesvol opgeslagen als {output_pad}")

In [0]:
display(df_pandas)